# Octane Access via API

In [2]:
import json
import urllib
import urllib3
import requests
import logging
from sso_session import bmw_sso_session
import numpy as np
import pandas as pd

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
with open("c:/Users/q446328/Desktop/TM_DM_Jupyter/login_info.txt", "r") as login_file:
    user_data = json.loads(login_file.read())
    user_name = user_data["username"]
    password = user_data["password"]

In [47]:
BASE_URL = "https://octane-prod.bmwgroup.net"
API_URL = "https://octane-prod.bmwgroup.net/api/shared_spaces/1002/workspaces/2001"
DEFECT_FINDER_TEAM = "DTSV_China"
EP_USER = "workspace_users"
EP_METADATA = "metadata/fields"
EP_DEFECT = "defects"
EP_MANUALRUN = 'manual_runs'
F_USER = ("name", "first_name", "last_name", "name", "id", "email")
F_DEFECT = ("id", 
            "name", 
            "creation_time",
            "last_modified",
            "parent_child_udf",
            "team",               
            "vin_udf",
            "user_tags",
            "tqr_udf", # ticket quality
            "product_areas", # AIDA
            "aida_businesskey_udf", 
            "software_version_udf",
            "ecu_no_of_changes_udf",
            "first_use_sop_of_function_udf",
            "tolerated_count_udf",
            "blocking_reason_udf",
            "reprel_changes_udf", 
            "parent", 
            "assigned_ecu_udf", 
            "error_occurrence_udf",
            "problem_finder_team_udf", 
            "program", 
            "solution_responsible_udf", 
            "detected_in_release", 
            "detected_by", 
            "function_responsible1_udf", 
            "owner", 
            "phase", 
            "severity", 
            "involved_i_step1_udf", 
            "author", 
            "lead_model_udf", 
            "problem_severity_udf", 
            "ecu_to_modul_udf"
           )
F_MANUALTEST = ('author', 'automation_status', 'covered_requirement', 'description', 'design_department_udf', 'designer', 'ecu_udf')
F_MANUALRUN = ("defect",
               "is_completed",
               "steps_num",
               "name",
               "version_stamp",
               "id",
               "last_modified",
               "started",
               "creation_time",
               "test_name",
               "test",
               "finished_udf",
               "testplatformid_udf",
               "exec_model_series_udf",
               "execution_sw_version_udf", # SW version
               "test",
               "author",
               "release",
               "run_by",
               "product_areas",
               "program",
               "set_udf",
               "testing_tool_type",
               "taxonomies",
               "test_version",
               "test_phase",
               "run_team_000_udf",
               "domain_udf",
               "status",
               "native_status",
               "target_ecu_conf_udf" # includes the platform information
              )

## Login

Using the `bmw_sso` library.

In [42]:
session = requests.Session()
session.proxies.update({"https": "160.48.211.81:8080"})
session = bmw_sso_session(BASE_URL, user_name, password, session=session)

Verify whehter the login is successful via checking the user data, `<Response [200]>` will be shown after successful login.

In [43]:
request_params = {"fields": ",".join(F_USER), "query": '"(name=\'{}\')"'.format(user_name)}
request_params = urllib.parse.urlencode(request_params)
request_url = "{}/{}?{}".format(API_URL, EP_USER, request_params)
if session != None:
    resp = session.get(request_url, verify=False, allow_redirects=True)
    print(resp)

<Response [200]>


All the metadata definition can be retrieved with the following API.

In [44]:
request_url = "{}/{}".format(API_URL, EP_METADATA)
if session != None:
    resp = session.get(request_url, verify=False, allow_redirects=True)
    print(resp)

<Response [200]>


## Get the tickets and save the data to 'json' or 'csv' for future analysis

In [ ]:
request_params = {"fields": ",".join(F_DEFECT),
                  "query": '"(problem_finder_team_udf={name=\'DTSV_China\'});(creation_time>\'2025-01-01T15:59:59Z\';creation_time<\'2025-12-31T16:00:00Z\')"',
                  "order_by": 'creation_time',
                  "limit": 7000
                 }
request_params = urllib.parse.urlencode(request_params)
request_url = "{}/{}?{}".format(API_URL, EP_DEFECT, request_params)
if session != None:
    resp = session.get(request_url, verify=False, allow_redirects=True)
    print(resp); print(resp.json()["total_count"])
    with open("defect/2025_defect.json", "wb") as f:
       f.write(resp.content)
    # pd.json_normalize(resp.json()['data'], sep='_').to_csv('2025_defect.csv', index=False)

<Response [200]>
1541


In [ ]:
request_params = {"fields": ",".join(F_DEFECT),
                  "query": '"(problem_finder_team_udf={name=\'DTSV_China\'});(creation_time>\'2024-01-01T15:59:59Z\';creation_time<\'2024-12-31T16:00:00Z\')"',
                  "order_by": 'creation_time',
                  "limit": 7000
                 }
request_params = urllib.parse.urlencode(request_params)
request_url = "{}/{}?{}".format(API_URL, EP_DEFECT, request_params)
if session != None:
    resp = session.get(request_url, verify=False, allow_redirects=True)
    print(resp); print(resp.json()["total_count"])
    with open("defect/2024_defect.json", "wb") as f:
       f.write(resp.content)
    #pd.json_normalize(resp.json()['data'], sep='_').to_csv('2024_defect.csv', index=False)

<Response [200]>
6704


In [ ]:
request_params = {"fields": ",".join(F_DEFECT),
                  "query": '"(problem_finder_team_udf={name=\'DTSV_China\'});(creation_time>\'2024-01-01T15:59:59Z\';creation_time<\'2024-12-31T16:00:00Z\')"',
                  "order_by": 'creation_time',
                  "limit": 7000
                 }
request_params = urllib.parse.urlencode(request_params)
request_url = "{}/{}?{}".format(API_URL, EP_DEFECT, request_params)
request_url

'https://octane-prod.bmwgroup.net/api/shared_spaces/1002/workspaces/2001/defects?fields=id%2Cname%2Ccreation_time%2Clast_modified%2Cparent_child_udf%2Cteam%2Cvin_udf%2Cuser_tags%2Ctqr_udf%2Cproduct_areas%2Caida_businesskey_udf%2Csoftware_version_udf%2Cecu_no_of_changes_udf%2Cfirst_use_sop_of_function_udf%2Ctolerated_count_udf%2Cblocking_reason_udf%2Creprel_changes_udf%2Cparent%2Cassigned_ecu_udf%2Cerror_occurrence_udf%2Cproblem_finder_team_udf%2Cprogram%2Csolution_responsible_udf%2Cdetected_in_release%2Cdetected_by%2Cfunction_responsible1_udf%2Cowner%2Cphase%2Cseverity%2Cinvolved_i_step1_udf%2Cauthor%2Clead_model_udf%2Cproblem_severity_udf%2Cecu_to_modul_udf&query=%22%28problem_finder_team_udf%3D%7Bname%3D%27DTSV_China%27%7D%29%3B%28creation_time%3E%272024-01-01T15%3A59%3A59Z%27%3Bcreation_time%3C%272024-12-31T16%3A00%3A00Z%27%29%22&order_by=creation_time&limit=7000'

## Manual Test and Manual Runs

### MR

Manual runs for year 2024

In [ ]:
# ('01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13')

for release in ('01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13'):
    release_query = "run_team_000_udf EQ {name EQ ^DTSV_China^};release EQ {name EQ ^R-24-" + release + "^}"
    request_params = {"fields": ','.join(F_MANUALRUN),
                      "query": '\"{}\"'.format(release_query),
                      'limit': 3500
                     }
    request_params = urllib.parse.urlencode(request_params)
    request_url = "{}/{}?{}".format(API_URL, EP_MANUALRUN, request_params)
    if session != None:
        resp = session.get(request_url, verify=False, allow_redirects=True)
        print(resp); print(resp.json()['total_count'])
        #pd.json_normalize(resp.json()['data'], sep='_').to_csv(f'mr/R24{}.csv'.format(release), index=False)
        
        with open(f'mr/R24{release}.json', 'w', encoding='utf-8') as json_file:
            json.dump(resp.json(), json_file, ensure_ascii=False, indent=4)

<Response [200]>
1801
<Response [200]>
2669
<Response [200]>
2125
<Response [200]>
2091
<Response [200]>
2004
<Response [200]>
2563
<Response [200]>
2209
<Response [200]>
2584
<Response [200]>
2214
<Response [200]>
2194
<Response [200]>
2983
<Response [200]>
2794
<Response [200]>
1518


Manual runs for year 2025

In [ ]:
# ('01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13')

for release in ('01', '02', '03', '04'):
    release_query = "run_team_000_udf EQ {name EQ ^DTSV_China^};release EQ {name EQ ^R-25-" + release + "^}"
    request_params = {"fields": ','.join(F_MANUALRUN),
                      "query": '\"{}\"'.format(release_query),
                      'limit': 3500
                     }
    request_params = urllib.parse.urlencode(request_params)
    request_url = "{}/{}?{}".format(API_URL, EP_MANUALRUN, request_params)
    if session != None:
        resp = session.get(request_url, verify=False, allow_redirects=True)
        print(resp); print(resp.json()['total_count'])
        with open(f"mr/R25{release}.json".format(release), "wb") as f:
            f.write(resp.content)
        #pd.json_normalize(resp.json()['data'], sep='_').to_csv('R25{}.csv'.format(release), index=False)

<Response [200]>
1895
<Response [200]>
2452
<Response [200]>
1690
<Response [200]>
0


In [ ]:
request_params = {"fields": ",".join(F_DEFECT),
                  "query": '"(problem_finder_team_udf={name=\'DTSV_China\'});(creation_time>\'2024-01-01T15:59:59Z\';creation_time<\'2024-12-31T16:00:00Z\')"',
                  "order_by": 'creation_time',
                  "limit": 7000
                 }
request_params = urllib.parse.urlencode(request_params)
request_url = "{}/{}?{}".format(API_URL, EP_DEFECT, request_params)

In [ ]:
request_url

In [ ]:
import json

import urllib

import urllib3

import requests

import os

import pandas as pd

import numpy as np

import time

import concurrent.futures

from datetime import datetime, timedelta

from tqdm import tqdm  # 用于显示进度条



# 禁用SSL警告

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



# API配置

BASE_URL = "https://octane-prod.bmwgroup.net"

API_URL = "https://octane-prod.bmwgroup.net/api/shared_spaces/1002/workspaces/2001"

EP_DEFECT = "defects"

EP_HISTORY = "history_logs"



# 需要获取的defect字段

F_DEFECT = ("id", "name", "last_modified", "creation_time", "team", "problem_finder_team_udf", "severity", "phase", "owner", "detected_in_release")



def safe_parse_datetime(date_str):

    """安全地解析不同格式的日期时间字符串"""

    if not date_str or not isinstance(date_str, str):

        return pd.NaT

   

    # 尝试不同的日期格式

    formats = [

        # 标准ISO格式

        "%Y-%m-%dT%H:%M:%S.%fZ",  # 2023-01-15T12:30:45.123Z

        "%Y-%m-%dT%H:%M:%SZ",     # 2023-01-15T12:30:45Z

        "%Y-%m-%dT%H:%M:%S",      # 2023-01-15T12:30:45

        "%Y-%m-%dT%H:%M:%S.%f",   # 2023-01-15T12:30:45.123

        "%Y-%m-%d %H:%M:%S",      # 2023-01-15 12:30:45

        "%Y-%m-%d"                # 2023-01-15

    ]

   

    for fmt in formats:

        try:

            # 去掉可能的时区标识

            cleaned_str = date_str.replace('Z', '')

            if '.' in cleaned_str:

                cleaned_str = cleaned_str.split('.')[0]

            return pd.to_datetime(cleaned_str, format=fmt)

        except (ValueError, TypeError):

            continue

   

    # 最后尝试pandas内置解析（较慢但更灵活）

    try:

        return pd.to_datetime(date_str, errors='coerce')

    except:

        return pd.NaT



def get_all_defect_ids(session, headers, team="DTSV_China", start_date=None, end_date=None, filter_field="creation_time"):

    """获取符合条件的所有defect ID

   

    Args:

        session: 请求会话

        headers: 请求头

        team: 团队名称

        start_date: 开始日期，格式为'YYYY-MM-DD'

        end_date: 结束日期，格式为'YYYY-MM-DD'

        filter_field: 筛选字段，'creation_time'或'last_modified'

       

    Returns:

        tuple: (defect_ids, defect_data)

    """

    all_defect_ids = []

    all_defect_data = []

    offset = 0

    limit = 1000

   

    # 格式化日期为API需要的格式

    start_date_formatted = f"{start_date}T00:00:00Z" if start_date else "2025-01-01T00:00:00Z"

    end_date_formatted = f"{end_date}T23:59:59Z" if end_date else "2025-12-31T23:59:59Z"

   

    field_description = "创建" if filter_field == "creation_time" else "修改"

    print(f"正在获取{team}团队在{start_date}至{end_date}期间{field_description}的defect列表...")

   

    while True:

        # 构建查询

        query = f'"(problem_finder_team_udf={{name=\'{team}\'}};{filter_field}>=\'{start_date_formatted}\';{filter_field}<=\'{end_date_formatted}\')"'

       

        request_params = {

            "fields": ",".join(F_DEFECT),

            "query": query,

            "limit": limit,

            "offset": offset

        }

       

        request_params = urllib.parse.urlencode(request_params)

        request_url = f"{API_URL}/{EP_DEFECT}?{request_params}"

       

        try:

            resp = session.get(request_url, verify=False, headers=headers)

            if resp.status_code >= 400:

                print(f"错误: {resp.status_code} - {resp.content}")

                break

               

            data = resp.json()

            batch_defects = data.get("data", [])

           

            # 存储完整的defect数据和ID

            all_defect_data.extend(batch_defects)

            batch_defect_ids = [item["id"] for item in batch_defects]

            all_defect_ids.extend(batch_defect_ids)

           

            print(f"本批次获取到 {len(batch_defects)} 个defects")

           

            # 如果返回的结果少于limit，说明已经获取完所有数据

            if len(batch_defects) < limit:

                break

               

            offset += limit

            print(f"目前总共获取到 {len(all_defect_ids)} 个defects")

           

            # 降低请求频率

            time.sleep(0.3)

           

        except Exception as e:

            print(f"获取defect ID时发生错误: {e}")

            break

   

    # 将完整的defect数据保存为JSON和CSV

    if all_defect_data:

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

       

        # 创建输出文件前缀

        file_prefix = f"{team}_{filter_field}_{start_date}_to_{end_date}_{timestamp}"

       

        # 保存为JSON

        with open(f"./history/defects_{file_prefix}.json", "w", encoding="utf-8") as f:

            json.dump(all_defect_data, f, ensure_ascii=False, indent=2)

       

        # 保存为CSV以便分析

        df = pd.json_normalize(all_defect_data)

        df.to_csv(f"./history/defects_{file_prefix}.csv", index=False, encoding="utf-8")

       

        print(f"已将{len(all_defect_data)}个defect的详细信息保存至文件")

   

    return all_defect_ids, all_defect_data



def get_defect_history(defect_id, session, headers):

    """获取单个defect的历史记录"""

    S_QUERY = f'"(entity_id=\'{defect_id}\';entity_type=\'defect\')"'

   

    request_params = {

        "query": S_QUERY,

        "limit": "max",

        "offset": 0,

        "order_by": "-timestamp"

    }

   

    request_params = urllib.parse.urlencode(request_params)

    request_url = f"{API_URL}/{EP_HISTORY}?{request_params}"

   

    try:

        resp = session.get(request_url, verify=False, headers=headers, allow_redirects=True)

       

        if not 200 <= resp.status_code < 400:

            return None

       

        return resp.json()

   

    except Exception as e:

        return None



def process_history_data(history_data, defect_id, defect_info=None):

    """处理历史数据，提取有意义的信息"""

    processed_entries = []

   

    if not history_data or "data" not in history_data:

        return processed_entries

   

    # 基础defect信息

    defect_base_info = {}

    if defect_info:

        for defect in defect_info:

            if defect["id"] == defect_id:

                defect_base_info = {

                    "defect_name": defect.get("name", ""),

                    "severity": defect.get("severity", {}).get("name", "") if isinstance(defect.get("severity"), dict) else "",

                    "phase": defect.get("phase", {}).get("name", "") if isinstance(defect.get("phase"), dict) else "",

                    "owner": defect.get("owner", {}).get("name", "") if isinstance(defect.get("owner"), dict) else "",

                    "creation_time": defect.get("creation_time", "")

                }

                break

       

    for entry in history_data["data"]:

        basic_entry = {

            "defect_id": defect_id,

            "timestamp": entry.get("timestamp"),

            **defect_base_info

        }

       

        # 添加用户信息

        if "workspace_user" in entry and entry["workspace_user"] is not None:

            basic_entry["user_name"] = entry["workspace_user"].get("name", "")

            basic_entry["user_id"] = entry["workspace_user"].get("id", "")

       

        # 处理修改字段

        if "modified_fields" in entry and entry["modified_fields"] is not None:

            for field in entry["modified_fields"]:

                field_entry = basic_entry.copy()

                field_entry["field_name"] = field.get("name", "unknown_field")

                field_entry["previous_value"] = str(field.get("previous_value", ""))

                field_entry["new_value"] = str(field.get("new_value", ""))

                processed_entries.append(field_entry)

        else:

            # 处理无修改字段的条目（例如评论）

            processed_entries.append(basic_entry)

   

    return processed_entries



def get_histories_parallel(defect_ids, session, headers, defect_data=None, max_workers=10):

    """并行获取多个defect的历史记录"""

    all_history_entries = []

    processed_ids = []

    error_ids = []

   

    # 创建一个进度条

    with tqdm(total=len(defect_ids), desc="获取历史记录") as pbar:

        # 使用线程池并行处理

        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:

            # 提交所有任务

            future_to_id = {

                executor.submit(get_defect_history, defect_id, session, headers): defect_id

                for defect_id in defect_ids

            }

           

            # 处理完成的任务

            for future in concurrent.futures.as_completed(future_to_id):

                defect_id = future_to_id[future]

                try:

                    history_data = future.result()

                   

                    if history_data:

                        # 保存单个历史记录

                        with open(f"./history/{defect_id}_history.json", "w", encoding="utf-8") as f:

                            json.dump(history_data, f, ensure_ascii=False, indent=2)

                       

                        # 处理历史数据

                        processed_entries = process_history_data(history_data, defect_id, defect_data)

                        all_history_entries.extend(processed_entries)

                        processed_ids.append(defect_id)

                    else:

                        error_ids.append(defect_id)

               

                except Exception as e:

                    print(f"\n处理defect {defect_id}时发生错误: {e}")

                    error_ids.append(defect_id)

               

                finally:

                    pbar.update(1)

   

    return all_history_entries, processed_ids, error_ids



def merge_defect_and_history_data(defect_data, history_entries):

    """合并defect数据和历史记录"""

    # 创建defect ID到defect数据的映射

    defect_map = {defect["id"]: defect for defect in defect_data}

   

    # 为每个历史条目添加完整的defect信息

    enriched_entries = []

    for entry in history_entries:

        defect_id = entry.get("defect_id")

        if defect_id in defect_map:

            # 添加感兴趣的defect字段

            defect_info = defect_map[defect_id]

            entry_with_defect = entry.copy()

           

            # 可以添加更多需要的defect字段

            if "name" in defect_info:

                entry_with_defect["defect_name"] = defect_info["name"]

               

            if "severity" in defect_info and isinstance(defect_info["severity"], dict):

                entry_with_defect["severity"] = defect_info["severity"].get("name", "")

               

            if "phase" in defect_info and isinstance(defect_info["phase"], dict):

                entry_with_defect["phase"] = defect_info["phase"].get("name", "")

               

            if "owner" in defect_info and isinstance(defect_info["owner"], dict):

                entry_with_defect["owner"] = defect_info["owner"].get("name", "")

           

            enriched_entries.append(entry_with_defect)

        else:

            enriched_entries.append(entry)

   

    return enriched_entries



def get_date_input(prompt, default=None):

    """获取日期输入，并验证格式"""

    while True:

        date_str = input(prompt)

        if not date_str and default:

            return default

       

        try:

            # 验证日期格式

            datetime.strptime(date_str, "%Y-%m-%d")

            return date_str

        except ValueError:

            print("日期格式不正确，请使用YYYY-MM-DD格式")



def main():

    # 创建session

    session = requests.Session()

   

    # 获取用户的cookie

    cookie = input("请复制Browser F12中的cookie: ")

    headers = {

        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36",

        "cookie": cookie

    }

   

    # 创建history目录

    os.makedirs("./history", exist_ok=True)

   

    # 时间戳用于文件名

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

   

    # 获取团队名称

    team = input("请输入要查询的团队名称(默认DTSV_China): ") or "DTSV_China"

   

    # 获取筛选方式

    print("\n请选择筛选方式:")

    print("1. 按创建时间筛选")

    print("2. 按最后修改时间筛选")

    filter_option = input("选择(默认1): ") or "1"

    filter_field = "creation_time" if filter_option == "1" else "last_modified"

    field_description = "创建" if filter_option == "1" else "修改"

   

    # 获取时间范围

    print(f"\n请输入要查询的{field_description}时间范围（格式为YYYY-MM-DD）:")

    # 默认为当前月份的第一天

    default_start = datetime.now().replace(day=1).strftime("%Y-%m-%d")

    # 默认为当天

    default_end = datetime.now().strftime("%Y-%m-%d")

   

    start_date = get_date_input(f"开始日期 (默认 {default_start}): ", default_start)

    end_date = get_date_input(f"结束日期 (默认 {default_end}): ", default_end)

   

    # 获取defect ID和完整defect数据

    defect_ids, defect_data = get_all_defect_ids(

        session, headers, team=team, start_date=start_date,

        end_date=end_date, filter_field=filter_field

    )

   

    if not defect_ids:

        print(f"没有找到{team}团队在{start_date}至{end_date}期间{field_description}的defects!")

        return

   

    print(f"找到了 {len(defect_ids)} 个符合条件的defects。")

   

    # 获取用户输入的并行数

    max_workers = int(input("请输入并行处理的线程数(默认5，最大20): ") or "5")

    max_workers = min(max(1, max_workers), 20)  # 确保在1-20之间

   

    print(f"正在使用{max_workers}个线程并行获取每个defect的历史记录...")

   

    # 并行获取历史记录

    history_entries, processed_ids, error_ids = get_histories_parallel(

        defect_ids, session, headers, defect_data, max_workers=max_workers

    )

   

    print(f"\n历史记录获取完成！成功: {len(processed_ids)}, 失败: {len(error_ids)}")

   

    # 保存获取失败的ID列表

    if error_ids:

        with open(f"./history/failed_defect_ids_{timestamp}.json", "w") as f:

            json.dump(error_ids, f)

        print(f"已将获取失败的{len(error_ids)}个defect ID保存至文件")

   

    # 合并defect数据和历史记录

    print("正在合并defect信息和历史记录...")

    enriched_history = merge_defect_and_history_data(defect_data, history_entries)

   

    # 保存处理后的历史记录

    if enriched_history:

        # 文件名前缀

        file_prefix = f"{team}_{filter_field}_{start_date}_to_{end_date}_{timestamp}"

       

        # 保存为JSON

        json_filename = f"./history/history_{file_prefix}.json"

        with open(json_filename, "w", encoding="utf-8") as f:

            json.dump(enriched_history, f, ensure_ascii=False, indent=2)

        print(f"已将历史数据保存为JSON: {json_filename}")

       

        # 转换为DataFrame并保存为CSV

        df = pd.DataFrame(enriched_history)

       

        # 对数据进行排序，便于分析

        if "timestamp" in df.columns:

            # 使用安全的日期解析方法

            df["timestamp"] = df["timestamp"].apply(safe_parse_datetime)

            df = df.sort_values(["defect_id", "timestamp"], ascending=[True, False])

           

        csv_filename = f"./history/history_{file_prefix}.csv"

        df.to_csv(csv_filename, index=False, encoding="utf-8")

        print(f"已将{len(enriched_history)}条历史记录保存到CSV文件: {csv_filename}")

       

        try:

            # 可选：生成Excel文件以便更好地查看

            excel_filename = f"./history/history_{file_prefix}.xlsx"

            df.to_excel(excel_filename, index=False, engine="openpyxl")

            print(f"已将历史记录保存到Excel文件: {excel_filename}")

        except Exception as e:

            print(f"保存Excel文件时出错: {e}")

    else:

        print("没有收集到历史数据!")

   

    print("处理完成!")



if __name__ == "__main__":

    main()

